# 🎙️ TurboVoiceCloner — Chatterbox Multilingual & Hindi SOTA Studio (Colab T4 Free)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/colab/08_f5_chatterbox_hindi_voice.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://github.com/SudhirDevOps1/t4-gpu-llm-api/blob/main/README.md#running-on-kaggle-manual-import)
[![GitHub Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **⚡ Architecture Highlights (TurboVoiceCloner + Resemble AI):**
> - 🎯 **Zero-Shot Voice Cloning:** Clones any voice from a 3–10 sec audio sample with exceptional timbre fidelity.
> - 🇮🇳 **Native Hindi (`hi`) Support:** Devanagari script, phonemic inflection, and natural cadence.
> - 🎭 **Emotion Exaggeration:** Precision sliders for emotion intensity (`0.25` - `2.0`), CFG guidance pace, and temperature.
> - 📚 **Long-Text Paragraph Synthesis:** Uses `sentencex` to segment unlimited text into natural phrases and stitches them with `pydub`.
> - ☕ **Anti-Sleep Keep-Alive:** Background audio loop prevents Google Colab from disconnecting during generation.
> - 🛡️ **Zero-Leak Security:** Reads tokens via memory only — zero commit leaks.


### ⚠️ **Usage Disclaimer**

Use of this voice cloning model is subject to strict ethical and legal standards. By using this tool, you agree **not to** engage in any of the following prohibited activities:

* **Fraud or Deception**: Using cloned voices to create misleading or fraudulent content.
* **Impersonation**: Replicating someone’s voice without their explicit permission.
* **Illegal Activities**: Violating local, national, or international laws.

> ⚖️ **Legal Responsibility**
> The developers disclaim all liability for misuse. **Users bear full responsibility** for ensuring their usage complies with all laws.

In [ ]:
#@title 🚀 Install Chatterbox Multilingual { display-mode: "form" }
import shutil
import os

%cd /content/

# 1. Completely clean any existing files to avoid conflicts
for file in os.listdir('/content/'):
    file_path = os.path.join('/content/', file)
    if file == '.config':
        continue
    try:
        if os.path.isdir(file_path):
            shutil.rmtree(file_path)
        else:
            os.remove(file_path)
    except Exception as e:
        pass

# 2. Re-download clean structure
!wget -q -O app.py https://raw.githubusercontent.com/NeuralFalconYT/Chatterbox-Multilingual/refs/heads/main/app.py
!git clone -q https://github.com/resemble-ai/chatterbox.git

# Patching system files
!rm -f /content/chatterbox/src/chatterbox/__init__.py
!wget -q -O /content/chatterbox/src/chatterbox/__init__.py https://raw.githubusercontent.com/NeuralFalconYT/Chatterbox-Multilingual/refs/heads/main/__init__.py

# 3. Force installation of pre-built binary packages to prevent any compilation hangs
!pip install --no-cache-dir --only-binary=:all: numpy==1.26.4
!pip install resampy==0.4.3 librosa==0.11.0 s3tokenizer==0.2.0 torchcodec==0.8.1 transformers==4.46.3 diffusers==0.29.0 resemble-perth==1.0.1 omegaconf==2.3.0 conformer==0.3.2 safetensors==0.5.3 sentencex==0.6.1 pydub==0.25.1 soundfile==0.13.1

from IPython.display import clear_output
import time

clear_output()
print("✅ Installation complete. The session will restart automatically...")
time.sleep(2)
os.kill(os.getpid(), 9)

In [ ]:
!pip install "numpy<2.3" "ml_dtypes>=0.4.0" hf_transfer

In [ ]:
#@title ☕ Prevent Colab Sleep { display-mode: "form" }
from IPython.display import Audio, display
print("Keep-alive audio is playing in the background.")
display(Audio("https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a", autoplay=True))

In [ ]:
#@title 🚀 Run Gradio App { display-mode: "form" }
%cd /content/
import os

# Uninstall conflicting torchvision to fix the operator error
print("🧹 Removing conflicting torchvision to fix the operator error...")
!pip uninstall -y -q torchvision

# Ensure sentencex and other lost dependencies are explicitly installed
print("📦 Installing missing dependencies (sentencex, conformer, etc.)...")
!pip install -q sentencex conformer resemble-perth s3tokenizer torchcodec

# Redownload app.py if it is missing
if not os.path.exists('app.py'):
    print("📥 Downloading app.py...")
    !wget -q https://raw.githubusercontent.com/NeuralFalconYT/Chatterbox-Multilingual/refs/heads/main/app.py

# Ensure the chatterbox library is installed locally
if os.path.exists('chatterbox'):
    print("📦 Installing local chatterbox library...")
    !pip install -q -e /content/chatterbox
else:
    print("⚠️ Warning: chatterbox directory not found! Cloning again...")
    !git clone -q https://github.com/resemble-ai/chatterbox.git
    !rm -f /content/chatterbox/src/chatterbox/__init__.py
    !wget -q -O /content/chatterbox/src/chatterbox/__init__.py https://raw.githubusercontent.com/NeuralFalconYT/Chatterbox-Multilingual/refs/heads/main/__init__.py
    !pip install -q -e /content/chatterbox

# Running the interface
# The third-party app has no password option, so do NOT publish it with --share.
# Instead we run it on localhost and open Colab's authenticated port proxy.
print("🚀 Starting the Gradio app on 127.0.0.1:7860 (this can take a few minutes)...")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import socket
import subprocess
import sys
import time

log = open("/tmp/chatterbox_app.log", "w")
subprocess.Popen([sys.executable, "app.py"], stdout=log, stderr=subprocess.STDOUT)

for _ in range(180):
    time.sleep(2)
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", 7860)) == 0:
            print("✅ Gradio is listening on 127.0.0.1:7860")
            break
else:
    print("⚠️ Port 7860 never opened. Last log lines:")
    print(open("/tmp/chatterbox_app.log").read()[-2000:])

try:
    from google.colab.output import serve_kernel_port_as_window
    print("🔐 Open this link (your Google login is required):")
    print(serve_kernel_port_as_window(7860))
except Exception as exc:
    print(f"Colab authenticated proxy unavailable: {exc}")
    print("Fallback only if you accept a public link: run `!python app.py --share` "
          "(this has NO password).")